In [0]:
%pip install torch transformers

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Company News → Sentiment Pipeline (Single Hop: Bronze → Silver Sentiment)
# MAGIC
# MAGIC This pipeline is now a single step:
# MAGIC bronze.company_news_polygon → (cleaning + FinBERT sentiment) → silver.news_sentiments
# MAGIC
# MAGIC There's no other intermediate table (the silver company_news_polygon table has
# MAGIC been removed from here). The incremental watermark now relies on
# MAGIC MAX(dwh_loaded_at) already present inside news_sentiments itself
# MAGIC (not a separate table), and dedup is now done on
# MAGIC (article_id, ticker) since that's the same merge key used by the final table.

# COMMAND ----------

# MAGIC %pip install -q transformers torch --upgrade
# MAGIC dbutils.library.restartPython()

# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import os
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("=" * 80)
print("Company News → Sentiment Pipeline (single hop)")
print("=" * 80)

# ==========================================================
# 0. Configuration
# ==========================================================

CATALOG = "finance_intelligence_hub"

BRONZE_TABLE            = f"{CATALOG}.bronze.company_news_polygon"
SILVER_SENTIMENT_TABLE  = f"{CATALOG}.silver.news_sentiments"

FINBERT_MODEL  = "ProsusAI/finbert"
MAX_TEXT_CHARS = 1024
BATCH_SIZE     = 16

# ==========================================================
# 1. Helper Functions (same ones used in the cleaning pipeline)
# ==========================================================

def blank_to_null(column):
    c = F.trim(column)
    return F.when(c == "", F.lit(None)).otherwise(c)


def clean_text(column):
    c = blank_to_null(column)
    c = F.regexp_replace(c, r"[\r\n\t]+", " ")
    c = F.regexp_replace(c, r"\s{2,}", " ")
    return F.trim(c)


def parse_related_tickers(column):
    c = blank_to_null(column)
    arr = F.split(F.coalesce(c, F.lit("")), ",")
    arr = F.transform(arr, lambda x: F.upper(F.trim(x)))
    arr = F.filter(arr, lambda x: x != "")
    arr = F.array_distinct(arr)
    return arr


def is_valid_url(column):
    c = blank_to_null(column)
    return c.rlike(r"^https?://")


def extract_author_email(column):
    c = blank_to_null(column)
    email = F.regexp_extract(c, r"([A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,})", 1)
    return F.when(email != "", email).otherwise(F.lit(None))


def extract_author_name(column):
    c = blank_to_null(column)
    without_email = F.trim(F.regexp_replace(c, r"^\S+@\S+", ""))
    name_in_parens = F.regexp_extract(without_email, r"\(([^)]+)\)", 1)
    return F.when(name_in_parens != "", name_in_parens).otherwise(c)


def parse_dwh_loaded_at(column):
    """
    Bronze has historically stored dwh_loaded_at in two different string
    formats ('2026-07-04T00:00:00Z' from the backfill, and
    '2026-07-04 00:12:33' from the live incremental script). try_to_timestamp
    returns NULL on a format mismatch instead of raising (unlike
    to_timestamp under ANSI mode), so trying both formats and coalescing
    is safe.
    """
    c = blank_to_null(column)
    iso_fmt = F.try_to_timestamp(c, F.lit("yyyy-MM-dd'T'HH:mm:ss'Z'"))
    plain_fmt = F.try_to_timestamp(c, F.lit("yyyy-MM-dd HH:mm:ss"))
    return F.coalesce(iso_fmt, plain_fmt)


# ==========================================================
# 2. Create Silver Sentiment Table (if it doesn't already exist)
#    Same columns as the news article + 3 score columns
# ==========================================================

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_SENTIMENT_TABLE}
(
article_id STRING,
ticker STRING,
related_tickers ARRAY<STRING>,
related_tickers_count INT,
title STRING,
description STRING,
text_for_finbert STRING,
text_word_count INT,
author STRING,
author_name STRING,
author_email STRING,
publisher_name STRING,
publisher_url STRING,
source_url STRING,
image_url STRING,
image_url_is_valid BOOLEAN,
published_at TIMESTAMP,
published_date_raw STRING,
dwh_loaded_at TIMESTAMP,
silver_loaded_at TIMESTAMP,
positive_score DOUBLE,
negative_score DOUBLE,
neutral_score DOUBLE
)
USING DELTA
""")

print("Silver sentiment table verified successfully.")

NEWS_COLUMNS = [
    "article_id", "ticker", "related_tickers", "related_tickers_count",
    "title", "description", "text_for_finbert", "text_word_count",
    "author", "author_name", "author_email",
    "publisher_name", "publisher_url", "source_url",
    "image_url", "image_url_is_valid",
    "published_at", "published_date_raw",
    "dwh_loaded_at", "silver_loaded_at",
]

# ==========================================================
# 3. Detect whether bronze has an ingestion-audit column
# ==========================================================

bronze_columns = spark.table(BRONZE_TABLE).columns
HAS_AUDIT_COL = "dwh_loaded_at" in bronze_columns

if HAS_AUDIT_COL:
    print("dwh_loaded_at column found -> running incremental load.")
else:
    print("WARNING: dwh_loaded_at column NOT found in bronze table.")
    print("Falling back to FULL LOAD every run (dedup + MERGE still keep it idempotent).")

# ==========================================================
# 4. Incremental Discovery
#    The watermark now relies on MAX(dwh_loaded_at) from news_sentiments
#    itself (there's no other intermediate table anymore)
# ==========================================================

if HAS_AUDIT_COL:

    last_loaded = spark.sql(
        f"SELECT COALESCE(MAX(dwh_loaded_at), TIMESTAMP('1900-01-01')) FROM {SILVER_SENTIMENT_TABLE}"
    ).first()[0]

    bronze_df = (
        spark.table(BRONZE_TABLE)
        # cast explicitly BEFORE filtering, so the watermark comparison
        # is timestamp-vs-timestamp, not string-vs-timestamp
        .withColumn("dwh_loaded_at", parse_dwh_loaded_at(F.col("dwh_loaded_at")))
        .filter(F.col("dwh_loaded_at") > last_loaded)
    )

else:

    bronze_df = spark.table(BRONZE_TABLE)
    bronze_df = bronze_df.withColumn("dwh_loaded_at", F.lit(None).cast("timestamp"))

rows_read = bronze_df.count()
print(f"Rows Read : {rows_read}")

if rows_read == 0:
    dbutils.notebook.exit("No new records.")

# ==========================================================
# 5. Data Cleaning
# This kind of table doesn't have business rules like numeric
# thresholds. What's needed here is "cleaning" not "rejection", so
# there's no quarantine table and no rows get dropped — exactly as agreed.
# ==========================================================

clean_df = (

    bronze_df

    #########################################################
    # Ticker / Related Tickers
    #########################################################
    .withColumn("ticker", F.upper(blank_to_null(F.col("ticker"))))
    .withColumn("related_tickers", parse_related_tickers(F.col("related_tickers")))
    .withColumn("related_tickers_count", F.size(F.col("related_tickers")))

    #########################################################
    # Text Fields (title / description / text_for_finbert)
    #########################################################
    .withColumn("title", clean_text(F.col("title")))
    .withColumn("description", clean_text(F.col("description")))
    .withColumn("text_for_finbert", clean_text(F.col("text_for_finbert")))
    .withColumn(
        "text_word_count",
        F.when(
            F.col("text_for_finbert").isNotNull(),
            F.size(F.split(F.col("text_for_finbert"), r"\s+"))
        ).otherwise(F.lit(0))
    )

    #########################################################
    # Author
    #########################################################
    .withColumn("author", blank_to_null(F.col("author")))
    .withColumn("author_name", extract_author_name(F.col("author")))
    .withColumn("author_email", extract_author_email(F.col("author")))

    #########################################################
    # Publisher / URLs
    #########################################################
    .withColumn("publisher_name", blank_to_null(F.col("publisher_name")))
    .withColumn("publisher_url", blank_to_null(F.col("publisher_url")))
    .withColumn("source_url", blank_to_null(F.col("source_url")))
    .withColumn("image_url", blank_to_null(F.col("image_url")))
    .withColumn("image_url_is_valid", is_valid_url(F.col("image_url")))

    #########################################################
    # Published Date
    # try_to_timestamp automatically returns NULL if the format is wrong
    # (without halting the code like a regular cast would)
    #########################################################
    .withColumn("published_date_raw", F.col("published_date"))
    .withColumn(
        "published_at",
        F.try_to_timestamp(F.col("published_date"), F.lit("yyyy-MM-dd'T'HH:mm:ss'Z'"))
    )

    #########################################################
    # Surrogate Key
    # article_id is stable per article, built from source_url
    # (or from title + published_date if there's no source_url)
    #########################################################
    .withColumn(
        "article_id",
        F.sha2(
            F.coalesce(
                F.col("source_url"),
                F.concat_ws("||", F.col("title"), F.col("published_date_raw"))
            ),
            256
        )
    )

    #########################################################
    # Audit
    #########################################################
    .withColumn("silver_loaded_at", F.current_timestamp())

)

print(f"Rows after cleaning : {clean_df.count()}")

# ==========================================================
# 6. Remove Duplicates
#    The final merge key is (article_id, ticker), so dedup is
#    done at that same grain, not on article_id alone
# ==========================================================

window_spec = (
    Window
    .partitionBy("article_id", "ticker")
    .orderBy(
        F.col("dwh_loaded_at").desc_nulls_last(),
        F.col("published_at").desc_nulls_last()
    )
)

silver_df = (
    clean_df
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .filter(F.col("ticker").isNotNull())
)

rows_valid = silver_df.count()
print(f"Rows After Dedup : {rows_valid}")

# ==========================================================
# 7. FinBERT Sentiment Scoring — directly on the same dataframe
#    (no intermediate table)
# ==========================================================

os.environ["HF_HOME"] = "/tmp/hf_cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

scoring_df = (
    silver_df
    .select(*NEWS_COLUMNS)
    .withColumn(
        "text_for_sentiment",  # temporary column, not stored in the final table
        F.substring(F.col("text_for_finbert"), 1, MAX_TEXT_CHARS)
    )
    .filter(F.length(F.coalesce(F.col("text_for_sentiment"), F.lit(""))) > 0)
)

pdf = scoring_df.toPandas()

texts_list = pdf["text_for_sentiment"].fillna("").astype(str).tolist()
total_news = len(texts_list)
total_batches = (total_news + BATCH_SIZE - 1) // BATCH_SIZE  # ceil division

print("=" * 80)
print(f"Total news to process : {total_news:,}")
print(f"Batch size            : {BATCH_SIZE}")
print(f"Total batches         : {total_batches:,}")
print("=" * 80)

tokenizer = AutoTokenizer.from_pretrained(FINBERT_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(FINBERT_MODEL)
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
id2label = model.config.id2label

pos_scores, neg_scores, neu_scores = [], [], []

for batch_num, start in enumerate(range(0, total_news, BATCH_SIZE), start=1):
    batch = texts_list[start:start + BATCH_SIZE]
    inputs = tokenizer(batch, padding=True, truncation=True, max_length=256, return_tensors="pt").to(device)

    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.nn.functional.softmax(logits, dim=-1).cpu().numpy()

    for row in probs:
        score_map = {id2label[i].lower(): float(row[i]) for i in range(len(row))}
        pos_scores.append(score_map.get("positive", 0.0))
        neg_scores.append(score_map.get("negative", 0.0))
        neu_scores.append(score_map.get("neutral", 0.0))

    del inputs, logits, probs

    news_done = min(start + BATCH_SIZE, total_news)
    print(f"Batch {batch_num:,}/{total_batches:,} done -> {news_done:,}/{total_news:,} news scored")

pdf["positive_score"] = pos_scores
pdf["negative_score"] = neg_scores
pdf["neutral_score"] = neu_scores

# Return exactly the same columns as news_sentiments: all article columns
# + the 3 scores (dropping the temporary text_for_sentiment column since
# it doesn't exist in the final table)
scored_pdf = pdf[NEWS_COLUMNS + ["positive_score", "negative_score", "neutral_score"]]

final_batch_df = spark.createDataFrame(scored_pdf)

rows_scored = final_batch_df.count()
print(f"Rows scored (ready for merge) : {rows_scored:,}")

# ==========================================================
# 8. MERGE INTO Silver Sentiment (the only place we write)
# ==========================================================

print("=" * 80)
print("Starting MERGE...")
print("=" * 80)

delta_table = DeltaTable.forName(spark, SILVER_SENTIMENT_TABLE)

(
    delta_table.alias("target")
    .merge(
        final_batch_df.alias("source"),
        "target.article_id = source.article_id AND target.ticker = source.ticker"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("MERGE completed successfully.")

# ==========================================================
# 9. Optimize Delta Table
# ==========================================================

print("Running OPTIMIZE...")
spark.sql(f"OPTIMIZE {SILVER_SENTIMENT_TABLE} ZORDER BY (ticker, published_at)")
print("Optimization completed.")

# ==========================================================
# 10. Pipeline Metrics
# ==========================================================

total_rows = spark.sql(f"SELECT COUNT(*) FROM {SILVER_SENTIMENT_TABLE}").first()[0]
total_articles = spark.sql(f"SELECT COUNT(DISTINCT article_id) FROM {SILVER_SENTIMENT_TABLE}").first()[0]

print()
print("=" * 80)
print("Pipeline Summary")
print("=" * 80)
print(f"Rows Read            : {rows_read:,}")
print(f"Rows After Dedup     : {rows_valid:,}")
print(f"Rows Scored/Merged   : {rows_scored:,}")
print(f"Distinct Articles    : {total_articles:,}")
print(f"Total Sentiment Rows : {total_rows:,}")
print("=" * 80)

In [0]:
%sql
drop table finance_intelligence_hub.gold.company_news_sentiment